[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philmui/worldmodels/blob/main/gait/skeleton-jepa/gavd/02-batch-extract-skeletons.ipynb)

# Part 2: Batch-extract skeletons from every sequence

We now have raw videos on disk. This notebook is where pixels become skeletons.
For every sequence in the manifest whose video downloaded, we open the video,
walk its annotated frames, and run MediaPipe BLAZEPOSE_33 to find 33 body joints
per frame. Stacking those per-frame skeletons gives one `(T, 33, 3)` array per
sequence, which is the raw material the JEPA trains on.

This mirrors the real batch extraction in the alexpose repo at
`experiments/exp5/01_extract_features.ipynb`, which loops conditions, loads each
sequence with `GAVDDataLoader`, and runs `SequenceKeypointExtractor` over its
frames. We keep the exact same pattern here and simply cache the raw joint arrays
instead of computing hand features, because the JEPA learns from the joints
directly rather than from 82 hand-designed numbers.

Two practical points matter at scale. First, the bounding box in each CSV tells
us where the walking person is. On the inline pose path we crop to that box
before running MediaPipe, which focuses the model on the right body, and then map
the joints back into whole-frame coordinates. Second, MediaPipe sometimes misses
a frame or returns too few joints, so we quality-filter, keeping only frames with
at least 25 of the 33 joints, and we record how each sequence fared.

## Run this locally or in Google Colab

In Colab, click the badge and run top to bottom. On your laptop, from the `gavd/`
folder:

```bash
uv sync --extra real     # the real path needs opencv-python and mediapipe
uv run jupyter lab 02-batch-extract-skeletons.ipynb
```

With `SMOKE_TEST = True`, the default, we synthesize a handful of walking
skeletons per condition, so you see the full extraction bookkeeping, the caching,
and an inline animation with no video or pose model needed. With
`SMOKE_TEST = False` we run MediaPipe over every downloaded sequence and cache the
real joint arrays. Set `MAX_SEQ_PER_CONDITION` to a small integer for a quick
real trial.

## From frame to joints

The figure shows the per-frame path: read the frame, use the bounding box to
focus on the walker, run BLAZEPOSE_33, and collect 33 joints. Repeating this over
a sequence's frames yields the `(T, 33, 3)` array we cache.

![Crop and pose](images/crop-and-pose.svg)

*Each annotated frame becomes 33 tracked joints; stacking frames gives one skeleton sequence per gait sequence.*

## Colab setup

In [ ]:
# Colab setup and local .env loading.
import importlib.util, shutil, subprocess, sys

_import_name = {"scikit-learn": "sklearn", "opencv-python": "cv2",
                "yt-dlp": "yt_dlp", "python-dotenv": "dotenv"}

def _ensure(pkgs):
    """pip install any packages whose import is not already available."""
    missing = [p for p in pkgs if importlib.util.find_spec(_import_name.get(p, p)) is None]
    if missing:
        print("Installing:", " ".join(missing))
        if importlib.util.find_spec("pip") is not None:
            cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing
        elif shutil.which("uv"):
            cmd = ["uv", "pip", "install", "-q"] + missing
        else:
            raise RuntimeError("Missing packages but neither pip nor uv is available: " + ", ".join(missing))
        subprocess.check_call(cmd)
    return missing

_ensure(["numpy", "pandas", "matplotlib", "python-dotenv", "tqdm"])

# The real path reads video and runs the pose model. Installing these up front
# keeps a fresh kernel from skipping them before CONFIG has been defined.
_ensure(["opencv-python", "mediapipe"])

from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())
print("Loaded environment via load_dotenv(find_dotenv()).")

ALEXPOSE_REPO = os.getenv("ALEXPOSE_REPO")
GAVD_DATA_DIR = os.getenv("GAVD_DATA_DIR")
YOUTUBE_CACHE_DIR = os.getenv("YOUTUBE_CACHE_DIR")
GAVD_CACHE_DIR = os.getenv("GAVD_CACHE_DIR")
if ALEXPOSE_REPO and os.path.isdir(ALEXPOSE_REPO) and ALEXPOSE_REPO not in sys.path:
    sys.path.insert(0, ALEXPOSE_REPO)
print("Setup complete.")

## Configuration

In [ ]:
from pathlib import Path

CONFIG = {
    "SMOKE_TEST": False,              # True -> synthesize skeletons, no video or pose model.
    "CACHE_DIR": Path(GAVD_CACHE_DIR) if GAVD_CACHE_DIR else Path.cwd() / "cache",
    "GAVD_DIR": Path(GAVD_DATA_DIR) if GAVD_DATA_DIR else Path.home() / "dev" / "alexpose" / "data" / "gavd",
    "YOUTUBE_DIR": Path(YOUTUBE_CACHE_DIR) if YOUTUBE_CACHE_DIR else Path.home() / "dev" / "alexpose" / "data" / "youtube",
    "POSE_LANDMARKER_MODEL": (Path(ALEXPOSE_REPO) / "data" / "models" / "pose_landmarker_full.task") if ALEXPOSE_REPO else Path("data/models/pose_landmarker_full.task"),
    "SUPPRESS_MEDIAPIPE_LOGS": True,
    "MAX_SEQ_PER_CONDITION": None,    # None -> every downloaded sequence. Set an int for a quick trial.
    "MIN_KEYPOINTS": 25,              # keep frames with at least this many of the 33 joints
    "C": 3,                           # channels per joint (x, y, z)
}
CONFIG["CACHE_DIR"].mkdir(parents=True, exist_ok=True)

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k:22s} = {v}")

## The 33 joints, the 35 edges, and the 6 groups

Before extracting anything, we pin down the skeleton definition we will use
everywhere in the series. BLAZEPOSE_33 tracks 33 named landmarks; the 35 edges
connect them into a body; and the 6 semantic groups (face, two arms, torso, two
legs) are how the JEPA hides whole limbs later. These constants are identical
across every notebook.

In [ ]:
import numpy as np

# 33 BLAZEPOSE_33 landmark names in index order (source: ambient/pose/keypoint_data.py)
LANDMARK_NAMES = [
    "NOSE","LEFT_EYE_INNER","LEFT_EYE","LEFT_EYE_OUTER","RIGHT_EYE_INNER",
    "RIGHT_EYE","RIGHT_EYE_OUTER","LEFT_EAR","RIGHT_EAR","MOUTH_LEFT",
    "MOUTH_RIGHT","LEFT_SHOULDER","RIGHT_SHOULDER","LEFT_ELBOW","RIGHT_ELBOW",
    "LEFT_WRIST","RIGHT_WRIST","LEFT_PINKY","RIGHT_PINKY","LEFT_INDEX",
    "RIGHT_INDEX","LEFT_THUMB","RIGHT_THUMB","LEFT_HIP","RIGHT_HIP",
    "LEFT_KNEE","RIGHT_KNEE","LEFT_ANKLE","RIGHT_ANKLE","LEFT_HEEL",
    "RIGHT_HEEL","LEFT_FOOT_INDEX","RIGHT_FOOT_INDEX",
]

# 35 skeleton edges as (i, j) index pairs
EDGES = [
    (0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(0,9),(0,10),(9,10),
    (11,12),(11,23),(12,24),(23,24),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]

# 6 semantic groups (for limb-based block masking)
GROUPS = {
    "face":      [0,1,2,3,4,5,6,7,8,9,10],
    "left_arm":  [11,13,15,17,19,21],
    "right_arm": [12,14,16,18,20,22],
    "torso":     [11,12,23,24],
    "left_leg":  [23,25,27,29,31],
    "right_leg": [24,26,28,30,32],
}
print(f"{len(LANDMARK_NAMES)} joints, {len(EDGES)} edges, {len(GROUPS)} semantic groups defined.")

## Animation and synthetic-skeleton helpers

So the motion behind the numbers is always visible, we define an inline animation
helper that draws the skeleton frame by frame and plays in both Jupyter and
Colab. We also define a synthetic walking-skeleton generator for smoke mode, so
this notebook has something to extract and animate without any video.

In [ ]:
# Inline animation helper: animate_skeleton
# Uses matplotlib.animation and to_jshtml so it works in Jupyter and Colab
# with no ffmpeg or system dependencies.
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate_skeleton(seq, edges, title="Walking skeleton", mask=None, fps=8):
    """Animate a (T, 33, C) skeleton sequence inline (uses x=seq[...,0], y=seq[...,1])."""
    T = seq.shape[0]
    x_all = seq[:, :, 0]; y_all = seq[:, :, 1]
    groups = [list(range(11)), [11,13,15,17,19,21], [12,14,16,18,20,22],
              [11,12,23,24], [23,25,27,29,31], [24,26,28,30,32]]
    colors = ["#8b5cf6", "#3b82f6", "#ef4444", "#22c55e", "#f59e0b", "#ec4899"]
    grey = "#cbd5e1"
    joint_mask = None
    if mask is not None:
        if mask.shape == (T, len(groups)):
            joint_mask = np.zeros((T, 33), dtype=bool)
            for t in range(T):
                for g_idx, grp in enumerate(groups):
                    if mask[t, g_idx]:
                        joint_mask[t, grp] = True
        else:
            joint_mask = mask
    fig, ax = plt.subplots(figsize=(6, 7))
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    x_min, x_max = x_all.min(), x_all.max()
    y_min, y_max = y_all.min(), y_all.max()
    margin = max(x_max - x_min, y_max - y_min) * 0.1
    def draw_frame(t):
        ax.clear(); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
        ax.set_xlim(x_min - margin, x_max + margin)
        ax.set_ylim(y_max + margin, y_min - margin)
        ax.set_title(f"{title} (frame {t}/{T})")
        x = x_all[t]; y = y_all[t]
        for g_idx, grp in enumerate(groups):
            grp_edges = [(i, j) for (i, j) in edges if i in grp and j in grp]
            for (i, j) in grp_edges:
                hidden = joint_mask is not None and (joint_mask[t, i] or joint_mask[t, j])
                c = grey if hidden else colors[g_idx]
                ax.plot([x[i], x[j]], [y[i], y[j]], color=c, linewidth=2, alpha=0.7)
        for g_idx, grp in enumerate(groups):
            x_g = [x[i] for i in grp]; y_g = [y[i] for i in grp]
            if joint_mask is not None:
                c_g = [grey if joint_mask[t, i] else colors[g_idx] for i in grp]
            else:
                c_g = [colors[g_idx]] * len(grp)
            ax.scatter(x_g, y_g, c=c_g, s=40, zorder=3, edgecolors='white', linewidths=0.5)
    anim = FuncAnimation(fig, draw_frame, frames=T, interval=1000/fps, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

def synthesize_walking_skeleton(T=16, seed=0, gait_bias=0.0):
    """A plausible synthetic (T, 33, 3) walking skeleton for SMOKE mode."""
    rng = np.random.RandomState(seed)
    base = np.zeros((33, 3), dtype=np.float32)
    # Full (x, y) layout for all 33 landmarks, laid out as a person seen head-on.
    # y grows downward (head near 0.16, feet near 0.97). x has the midline at 0.50,
    # with the left side (odd joint indices) left of it and the right side right of it.
    # Shoulders are wider than the hips, the arms hang OUTSIDE the hips down to about
    # hip height, and the head sits just above the shoulders, so the figure reads as a
    # real walking body instead of collapsing onto one vertical line.
    xs = {
        0:0.500,                                            # nose
        1:0.485, 2:0.475, 3:0.465, 4:0.515, 5:0.525, 6:0.535,  # eyes (left then right)
        7:0.455, 8:0.545,                                   # ears
        9:0.485, 10:0.515,                                  # mouth
        11:0.415, 12:0.585,                                 # shoulders (wide)
        13:0.395, 14:0.605,                                 # elbows (arms hang outside)
        15:0.405, 16:0.595,                                 # wrists
        17:0.395, 18:0.605, 19:0.405, 20:0.595, 21:0.420, 22:0.580,  # hands track their wrist
        23:0.455, 24:0.545,                                 # hips (narrower than shoulders)
        25:0.450, 26:0.550,                                 # knees
        27:0.448, 28:0.552,                                 # ankles
        29:0.448, 30:0.552, 31:0.455, 32:0.545,             # heels, foot tips
    }
    ys = {
        0:0.16,                                             # nose
        1:0.145, 2:0.145, 3:0.145, 4:0.145, 5:0.145, 6:0.145,  # eyes
        7:0.155, 8:0.155,                                   # ears
        9:0.185, 10:0.185,                                  # mouth (short neck to shoulders)
        11:0.24, 12:0.24,                                   # shoulders
        13:0.38, 14:0.38,                                   # elbows
        15:0.51, 16:0.51,                                   # wrists (about hip height)
        17:0.545, 18:0.545, 19:0.545, 20:0.545, 21:0.535, 22:0.535,  # hands (just past wrists)
        23:0.50, 24:0.50,                                   # hips
        25:0.71, 26:0.71,                                   # knees
        27:0.92, 28:0.92,                                   # ankles
        29:0.94, 30:0.94, 31:0.965, 32:0.965,               # heels, foot tips
    }
    for j in range(33):
        base[j, 0] = xs[j]
        base[j, 1] = ys[j]
    seq = np.repeat(base[None], T, axis=0)
    t = np.linspace(0, 2*np.pi, T, endpoint=False)
    swing = 0.06 * np.sin(t)
    for k, amp in [(25,1.0),(27,1.3),(31,1.4),(13,-0.8),(15,-1.0)]:
        seq[:, k, 0] += swing * amp * (1.0 + gait_bias)
    for k, amp in [(26,-1.0),(28,-1.3),(32,-1.4),(14,0.8),(16,1.0)]:
        seq[:, k, 0] += swing * amp * (1.0 - gait_bias)
    seq += rng.randn(T, 33, 3).astype(np.float32) * 0.004
    return seq.astype(np.float32)

print("animate_skeleton and synthesize_walking_skeleton defined.")

## The extraction function

The function below produces one `(T, 33, 3)` skeleton array for a single
sequence. In real mode it uses the alexpose `SequenceKeypointExtractor` when
`ambient` is importable, exactly as exp5 does: it takes the sequence DataFrame and
the video folder and returns a list of per-frame keypoint sets, which we convert
to an array. If `ambient` is unavailable, it falls back to reading frames with
OpenCV and running MediaPipe directly.

Two details matter for the joints to be usable downstream. First, we emit the
same coordinate convention on both paths: x and y normalized to the frame in
`[0, 1]`, and z as the raw MediaPipe depth. The alexpose `KeypointSet` stores
pixels in `.x` and `.y`, so we read its `.x_normalized` and `.y_normalized`
fields (or divide by the frame size) to match the inline path, otherwise the two
paths would feed notebook 03 skeletons on different scales. Second, on the inline
path we use the bounding box from the CSV to crop to the walking person before
running the pose model, which is the classic way to focus MediaPipe on the right
body, and we then map the joints back into whole-frame normalized coordinates so
the two paths still agree. If neither real path is available, the caller
synthesizes a skeleton so the notebook keeps running.

In [ ]:
import ast, contextlib, os, sys

def _kp_norm(kp):
    """Read one KeypointSet keypoint as (x, y, z) with x, y normalized to [0, 1].
    alexpose stores pixels in .x/.y and the [0,1] values in .x_normalized/.y_normalized."""
    xn = getattr(kp, "x_normalized", None)
    yn = getattr(kp, "y_normalized", None)
    if xn is None or yn is None:
        # Fall back to pixels divided by frame size if only .x/.y are present.
        return getattr(kp, "x", 0.0), getattr(kp, "y", 0.0), getattr(kp, "z", 0.0)
    return xn, yn, getattr(kp, "z", 0.0)

def keypointsets_to_array(kps_list, n_channels=3):
    """Convert a list of KeypointSet (each with .keypoints of 33) to (T, 33, C),
    with x, y normalized to [0, 1] and z the raw MediaPipe depth."""
    frames = []
    for ks in kps_list:
        pts = getattr(ks, "keypoints", None)
        if pts is None or len(pts) < 33:
            continue
        fw = float(getattr(ks, "frame_width", 0) or 0)
        fh = float(getattr(ks, "frame_height", 0) or 0)
        frame = np.zeros((33, n_channels), dtype=np.float32)
        for j in range(33):
            x, y, z = _kp_norm(pts[j])
            # If we only had pixels and know the frame size, normalize them here.
            if getattr(pts[j], "x_normalized", None) is None and fw > 0 and fh > 0:
                x, y = x / fw, y / fh
            frame[j, 0] = x
            frame[j, 1] = y
            if n_channels >= 3:
                frame[j, 2] = z
        frames.append(frame)
    return np.stack(frames, axis=0) if frames else np.zeros((0, 33, n_channels), np.float32)

def find_pose_landmarker_model(cfg):
    """Find the MediaPipe Tasks pose model used by the inline fallback."""
    candidates = [
        cfg.get("POSE_LANDMARKER_MODEL"),
        Path(ALEXPOSE_REPO) / "data" / "models" / "pose_landmarker_full.task" if ALEXPOSE_REPO else None,
        Path(ALEXPOSE_REPO) / "data" / "models" / "pose_landmarker_lite.task" if ALEXPOSE_REPO else None,
        Path.cwd() / "data" / "models" / "pose_landmarker_full.task",
        Path.cwd() / "data" / "models" / "pose_landmarker_lite.task",
    ]
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        path = Path(candidate).expanduser()
        checked.append(str(path))
        if path.exists():
            return path
    raise FileNotFoundError("No MediaPipe pose landmarker model found. Checked: " + ", ".join(checked))

@contextlib.contextmanager
def suppress_native_stderr(enabled=True):
    """Temporarily silence C/C++ libraries that write directly to the stderr file
    descriptor. This covers both MediaPipe's clearcut upload notices and ffmpeg's
    h264 decode chatter ('Missing reference picture', 'mmco: unref short failure'),
    which OpenCV's VideoCapture surfaces when it seeks into inter-coded frames."""
    if not enabled:
        yield
        return
    try:
        fd = sys.stderr.fileno()
    except Exception:
        with open(os.devnull, "w") as devnull, contextlib.redirect_stderr(devnull):
            yield
        return

    saved_fd = os.dup(fd)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), fd)
            yield
    finally:
        os.dup2(saved_fd, fd)
        os.close(saved_fd)

_INLINE_POSE_DETECTORS = {}

def get_inline_pose_detector(cfg, mp):
    """Return a cached MediaPipe detector callable and a report note."""
    suppress_logs = cfg.get("SUPPRESS_MEDIAPIPE_LOGS", True)
    if hasattr(mp, "solutions"):
        key = ("solutions",)
        if key not in _INLINE_POSE_DETECTORS:
            with suppress_native_stderr(suppress_logs):
                pose = mp.solutions.pose.Pose(static_image_mode=False, model_complexity=1)
            def detect_pose(rgb, _pose=pose):
                with suppress_native_stderr(suppress_logs):
                    result = _pose.process(rgb)
                return result.pose_landmarks.landmark if result.pose_landmarks else None
            _INLINE_POSE_DETECTORS[key] = (
                detect_pose,
                "inline mediapipe solutions (bbox crop)",
                pose.close,
            )
        detect_pose, inline_note, _ = _INLINE_POSE_DETECTORS[key]
        return detect_pose, inline_note

    from mediapipe.tasks import python as mp_tasks
    from mediapipe.tasks.python import vision
    model_path = find_pose_landmarker_model(cfg)
    key = ("tasks", str(model_path))
    if key not in _INLINE_POSE_DETECTORS:
        with suppress_native_stderr(suppress_logs):
            options = vision.PoseLandmarkerOptions(
                base_options=mp_tasks.BaseOptions(model_asset_path=str(model_path)),
                running_mode=vision.RunningMode.IMAGE,
                num_poses=1,
            )
            landmarker = vision.PoseLandmarker.create_from_options(options)
        def detect_pose(rgb, _landmarker=landmarker):
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            with suppress_native_stderr(suppress_logs):
                result = _landmarker.detect(mp_image)
            return result.pose_landmarks[0] if result.pose_landmarks else None
        _INLINE_POSE_DETECTORS[key] = (
            detect_pose,
            f"inline mediapipe tasks ({model_path.name}, bbox crop)",
            landmarker.close,
        )
    detect_pose, inline_note, _ = _INLINE_POSE_DETECTORS[key]
    return detect_pose, inline_note

def close_inline_pose_detectors():
    """Release cached MediaPipe detectors after a batch run."""
    for _, _, close_fn in list(_INLINE_POSE_DETECTORS.values()):
        try:
            close_fn()
        except Exception:
            pass
    _INLINE_POSE_DETECTORS.clear()

def bbox_scale_for_frame(row, W, H):
    """Ratio to map bbox pixels onto the DECODED frame (W, H).

    GAVD stores each bbox in the video's ORIGINAL resolution, recorded in the
    row's `vid_info` (commonly 1280x720). yt-dlp, however, often downloads a
    smaller stream (e.g. 640x360). Applying the raw bbox to the smaller frame
    crops the wrong region, MediaPipe finds no body, and every sequence yields
    zero frames. Scaling by decoded/original realigns the crop with the walker.
    Returns (1.0, 1.0) when `vid_info` is missing or unparseable."""
    try:
        vi = ast.literal_eval(row["vid_info"])
        ow = float(vi.get("width", 0) or 0)
        oh = float(vi.get("height", 0) or 0)
        if ow > 0 and oh > 0:
            return W / ow, H / oh
    except Exception:
        pass
    return 1.0, 1.0

def extract_sequence_real(csv_path, cfg):
    """Extract one sequence's (T, 33, 3) skeleton (x, y in [0,1], z raw) via ambient,
    falling back to an inline bbox-cropped MediaPipe pass."""
    # Preferred path: alexpose ambient (the exp5 pattern).
    try:
        from ambient.gavd import GAVDDataLoader
        from ambient.pose.keypoint_extractor import SequenceKeypointExtractor
        df = GAVDDataLoader().load_gavd_data(str(csv_path))
        ex = SequenceKeypointExtractor()
        kps = ex.extract_from_sequence(sequence_data=df, video_base_path=cfg["YOUTUBE_DIR"],
                                       filter_empty=True, min_keypoints=cfg["MIN_KEYPOINTS"],
                                       verbose=False)
        return keypointsets_to_array(kps, cfg["C"]), "ambient"
    except Exception as e:
        note = f"ambient path failed ({e}); trying inline MediaPipe"
    # Inline fallback: pandas + OpenCV + MediaPipe on the bbox crop.
    try:
        import pandas as pd, cv2, mediapipe as mp
        df = pd.read_csv(csv_path)
        video_id = str(df["id"].iloc[0])
        video_path = cfg["YOUTUBE_DIR"] / f"{video_id}.mp4"
        if not video_path.exists():
            return np.zeros((0, 33, cfg["C"]), np.float32), f"no video {video_id}.mp4"
        with suppress_native_stderr(cfg.get("SUPPRESS_MEDIAPIPE_LOGS", True)):
            cap = cv2.VideoCapture(str(video_path))
        detect_pose, inline_note = get_inline_pose_detector(cfg, mp)
        frames = []
        n_bbox_fallback = 0        # frames where the (scaled) bbox was unusable
        try:
            for _, r in df.iterrows():
                fn = int(r["frame_num"]) - 1              # frame_num is 1-based
                with suppress_native_stderr(cfg.get("SUPPRESS_MEDIAPIPE_LOGS", True)):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, fn))
                    ok, img = cap.read()
                if not ok:
                    continue
                H, W = img.shape[:2]
                # Crop to the bounding box so MediaPipe focuses on the walker. The
                # bbox is stored in the ORIGINAL resolution (vid_info), so scale it
                # onto the decoded (W, H) first, otherwise we crop the wrong region.
                try:
                    sx, sy = bbox_scale_for_frame(r, W, H)
                    b = ast.literal_eval(r["bbox"])
                    x0 = max(0, int(b["left"] * sx)); y0 = max(0, int(b["top"] * sy))
                    x1 = min(W, int((b["left"] + b["width"]) * sx))
                    y1 = min(H, int((b["top"] + b["height"]) * sy))
                except Exception:
                    x0, y0, x1, y1 = 0, 0, W, H          # no usable bbox: use the whole frame
                if x1 - x0 < 8 or y1 - y0 < 8:
                    x0, y0, x1, y1 = 0, 0, W, H          # degenerate box: fall back to full frame
                    n_bbox_fallback += 1
                crop = img[y0:y1, x0:x1]
                lm = detect_pose(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                if not lm:
                    continue
                # MediaPipe returns coords normalized to the CROP; map back to whole-frame [0,1].
                frame = np.zeros((33, cfg["C"]), np.float32)
                cw, ch = (x1 - x0), (y1 - y0)
                for j in range(33):
                    p = lm[j]
                    frame[j, 0] = (x0 + p.x * cw) / W
                    frame[j, 1] = (y0 + p.y * ch) / H
                    if cfg["C"] >= 3:
                        frame[j, 2] = p.z
                frames.append(frame)
        finally:
            cap.release()
        arr = np.stack(frames, 0) if frames else np.zeros((0, 33, cfg["C"]), np.float32)
        note = inline_note if not n_bbox_fallback else f"{inline_note}; {n_bbox_fallback} full-frame fallbacks"
        return arr, note
    except Exception as e:
        return np.zeros((0, 33, cfg["C"]), np.float32), f"{note}; inline failed ({e})"

print("extract_sequence_real defined.")

## Build the list of sequences to extract

We load the manifest and, in real mode, the download report, then keep only the
sequences whose video landed on disk. In smoke mode we fabricate a small set of
sequences across a few conditions so the loop has work to do. Grouping by
condition lets us cache one file per condition, which keeps the corpus tidy.

In [ ]:
import pandas as pd

manifest_path = CONFIG["CACHE_DIR"] / "manifest.csv"
report_path = CONFIG["CACHE_DIR"] / "download_report.csv"

def synthetic_todo():
    todo = []
    for cond, n, bias in [("normal", 3, 0.0), ("parkinsons", 2, 0.25),
                          ("stroke", 2, -0.3), ("myopathic", 2, 0.1), ("abnormal", 3, 0.05)]:
        for i in range(n):
            todo.append({"condition": cond, "seq": f"cl{cond[:3]}{i:04d}synthetic0000000",
                         "video_id": f"vid{i:05d}", "gait_bias": bias, "seed": hash((cond, i)) % 1000})
    return pd.DataFrame(todo)

if CONFIG["SMOKE_TEST"]:
    todo = synthetic_todo()
    print(f"SMOKE mode: {len(todo)} synthetic sequences across "
          f"{todo['condition'].nunique()} conditions.")
else:
    if not manifest_path.exists():
        raise FileNotFoundError("manifest.csv missing; run notebook 00 first.")
    manifest = pd.read_csv(manifest_path)
    if report_path.exists():
        report = pd.read_csv(report_path)
        ok_ids = set(report.loc[report["ok"], "video_id"].astype(str))
        manifest = manifest[manifest["video_id"].astype(str).isin(ok_ids)]
        print(f"Keeping {len(manifest)} sequences whose video downloaded.")
    else:
        print("No download_report.csv; attempting all manifest sequences.")
    if CONFIG["MAX_SEQ_PER_CONDITION"] is not None:
        manifest = manifest.groupby("condition", group_keys=False).head(CONFIG["MAX_SEQ_PER_CONDITION"])
    todo = manifest
print(todo.head())

## Run the batch extraction

We loop over every sequence, extract its skeleton, and stack the results into one
array per condition. We record the outcome for each sequence, its frame count
after filtering and any note, so the report tells us clearly which sequences
produced usable data. The progress bar makes a long real run legible.

In [ ]:
from tqdm.auto import tqdm

by_condition = {}      # condition -> list of (seq_id, (T,33,3) array)
records = []

for _, row in tqdm(todo.iterrows(), total=len(todo), desc="sequences"):
    cond = row["condition"]; seq_id = row["seq"]
    if CONFIG["SMOKE_TEST"]:
        T = 24
        arr = synthesize_walking_skeleton(T=T, seed=int(row["seed"]), gait_bias=float(row["gait_bias"]))
        note = "smoke synthetic"
    else:
        csv_path = CONFIG["GAVD_DIR"] / cond / f"{seq_id}.csv"
        arr, note = extract_sequence_real(csv_path, CONFIG)
    ok = arr.shape[0] >= 8      # need a few frames to be useful
    if ok:
        by_condition.setdefault(cond, []).append((seq_id, arr))
    records.append({"condition": cond, "seq": seq_id, "n_frames": int(arr.shape[0]),
                    "ok": bool(ok), "note": note})

if not CONFIG["SMOKE_TEST"]:
    close_inline_pose_detectors()

report_df = pd.DataFrame(records)
print(f"\nUsable sequences: {int(report_df['ok'].sum())}/{len(report_df)}")
print(report_df.groupby("condition")["ok"].agg(["sum", "count"]))

## Watch one extracted skeleton walk

Numbers are easier to trust when you can see them move. The animation below plays
one extracted sequence so you can watch the tracked body walk. In smoke mode this
is a synthetic skeleton; in real mode it is the joints MediaPipe found in the
video. This is the exact kind of motion the JEPA will learn to fill in.

In [ ]:
# Pick the first usable sequence to animate.
example_seq = None
for cond, items in by_condition.items():
    if items:
        example_seq = items[0][1]; example_cond = cond; break

if example_seq is not None and example_seq.shape[0] >= 2:
    print(f"Animating a '{example_cond}' sequence with {example_seq.shape[0]} frames.")
    display(animate_skeleton(example_seq[:24], EDGES, title=f"Extracted {example_cond} skeleton"))
else:
    print("No usable sequence to animate (check the extraction report above).")

## Cache one .npz per condition

We save the extracted skeletons grouped by condition, one `skeletons_<condition>.npz`
file each. Because sequences have different lengths, we store them as an object
array of `(T, 33, 3)` arrays plus a parallel list of sequence ids. Notebook 03
loads all of these, normalizes them, and slices them into the fixed-length windows
that make up the pretraining corpus.

In [ ]:
saved = []
for cond, items in by_condition.items():
    if not items:
        continue
    seq_ids = np.array([s for s, _ in items], dtype=object)
    arrays = np.empty(len(items), dtype=object)
    for i, (_, a) in enumerate(items):
        arrays[i] = a.astype(np.float32)
    safe = cond.replace(" ", "_")
    out = CONFIG["CACHE_DIR"] / f"skeletons_{safe}.npz"
    np.savez(out, seq_ids=seq_ids, arrays=arrays, condition=cond)
    saved.append((cond, len(items), out.name))

report_df.to_csv(CONFIG["CACHE_DIR"] / "extraction_report.csv", index=False)
print("Saved skeleton caches:")
for cond, n, name in saved:
    print(f"  {cond:16s}: {n} sequences -> {name}")
print(f"\nWrote extraction_report.csv with {len(report_df)} rows.")

## Recap and what comes next

We ran the batch extraction that turns downloaded videos into cached skeleton
sequences, one `(T, 33, 3)` array per gait sequence, grouped into a small set of
per-condition files. We quality-filtered along the way and saved a report of what
worked, and we watched one extracted skeleton walk.

In notebook 03 we load every cached sequence, normalize each one so the camera
position and scale stop mattering, and slice them into overlapping fixed-length
windows. That produces the large unlabeled clip bank the JEPA pretrains on, plus
the small labeled holdout we reserve for the final probe.